In [ ]:
!pip install -q openai-whisper pydub datasets huggingface_hub soundfile tqdm sarvamai striprtf
!apt-get install -qq ffmpeg

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
files = os.listdir("/content/drive/MyDrive/audiobook_eng")
print(f"found {len(files)} files:")
for f in sorted(files):
    print("  ", f)

In [ ]:

import os, json, re, time
from striprtf.striprtf import rtf_to_text
from pydub import AudioSegment
from tqdm import tqdm

INPUT_DIR  = "/content/drive/MyDrive/audiobook_eng/"
OUTPUT_DIR = "/content/eng_wavs/"
MANIFEST   = "/content/eng_manifest.json"
HF_TOKEN    = ""
HF_REPO     = "Kalpeshkhare7777/marathi-tts-audiobook"
TOTAL_SEGS = 17

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
manifest = []
skipped  = []

for i in tqdm(range(1, TOTAL_SEGS + 1), desc="processing"):
    rtf_path = f"{INPUT_DIR}seg{i}.rtf"
    mp3_path = f"{INPUT_DIR}seg{i}.mp3"
    wav_path = f"{OUTPUT_DIR}en_{i:03d}.wav"   # en_ prefix

    if not os.path.exists(rtf_path) or not os.path.exists(mp3_path):
        print(f" missing: seg{i} — skipping")
        skipped.append(i)
        continue

    with open(rtf_path, encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    text = rtf_to_text(raw)
    text = re.sub(r'\s+', ' ', text).strip()

    if len(text.split()) < 5:
        print(f" seg{i} text too short — skipping")
        skipped.append(i)
        continue

    audio = AudioSegment.from_mp3(mp3_path)
    audio = audio.set_channels(1).set_frame_rate(22050)
    audio.export(wav_path, format="wav")
    duration = len(audio) / 1000

    manifest.append({
        "id":           f"en_{i:03d}",   # en_ prefix
        "audio_path":   wav_path,
        "transcript":   text,
        "duration_sec": round(duration, 2),
        "language":     "en",            # English
        "emotion":      "",
        "label_source": "",
    })

with open(MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f" {len(manifest)} segments ready  |  {len(skipped)} skipped")
print(f"   total duration: {sum(m['duration_sec'] for m in manifest)/60:.1f} min")
for m in manifest[:3]:
    print(f"   {m['id']}  {m['duration_sec']}s  '{m['transcript'][:70]}'")




In [ ]:
from sarvamai import SarvamAI

client = SarvamAI(api_subscription_key=" ")

test = client.chat.completions(
    model="sarvam-30b",
    messages=[{"role": "user", "content": "reply with just the word: neutral"}]
)
try:
    content = test.choices[0].message.content
    print("dot notation →", content)
except AttributeError:
    content = test["choices"][0]["message"]["content"]
    print(" dict notation →", content)

In [ ]:
VALID_TAGS = [
    "narrative_calm", "sad", "happy", "excited",
    "tense", "angry", "formal", "conversational",
    "emotional", "neutral"
]

def parse_content(response):
    try:
        return response.choices[0].message.content
    except (AttributeError, IndexError):
        try:
            return response["choices"][0]["message"]["content"]
        except (KeyError, IndexError, TypeError):
            return None

def get_emotion(text, retries=3):
    prompt = (
        "You are labeling English audiobook text for TTS training.\n"   # updated for English
        "Choose exactly ONE tag from this list:\n"
        f"{', '.join(VALID_TAGS)}\n\n"
        "Rules:\n"
        "- Return ONLY the tag word, nothing else\n"
        "- No punctuation, no explanation\n\n"
        "Text:\n" + text[:500]
    )
    for attempt in range(retries):
        try:
            response = client.chat.completions(
                model="sarvam-30b",
                messages=[{"role": "user", "content": prompt}],
            )
            content = parse_content(response)
            if not content:
                raise ValueError("empty response from API")
            tag = content.strip().lower().strip('.,!?:;"\' ')
            return tag if tag in VALID_TAGS else "neutral"
        except Exception as e:
            print(f"   attempt {attempt+1}/{retries} failed: {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    return "neutral"

errors = 0
for item in tqdm(manifest, desc="tagging"):
    try:
        item["emotion"]      = get_emotion(item["transcript"])
        item["label_source"] = "sarvam-30b-llm"
        print(f"  {item['id']} → {item['emotion']}")
    except Exception as e:
        print(f"  error on {item['id']}: {e}")
        item["emotion"]      = "neutral"
        item["label_source"] = "fallback"
        errors += 1
    time.sleep(0.4)

with open(MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print(f"tagging done  |  {errors} errors")

In [ ]:

from collections import Counter

total_dur = sum(m["duration_sec"] for m in manifest)
emotions  = Counter(m["emotion"] for m in manifest)

print("━"*48)
print(f"  clips    : {len(manifest)}")
print(f"  total    : {total_dur/60:.1f} min")
print(f"  avg clip : {total_dur/len(manifest):.1f} sec")
print("━"*48)
print("  emotion distribution:")
for tag, cnt in emotions.most_common():
    bar = "▪" * cnt
    print(f"    {tag:20s} {cnt:2d}  {bar}")
print("━"*48)

for m in manifest:
    flags = []
    if m["duration_sec"] < 5:             flags.append("too short")
    if m["duration_sec"] > 120:           flags.append("too long")
    if len(m["transcript"].split()) < 5:  flags.append("short text")
    if flags:
        print(f"   {m['id']}  {m['duration_sec']}s  {flags}")
        print(f"       '{m['transcript'][:80]}'")

def fix_tag(clip_id, new_emotion):
    for m in manifest:
        if m["id"] == clip_id:
            old = m["emotion"]
            m["emotion"]      = new_emotion
            m["label_source"] = "human"
            print(f"{clip_id}: {old} → {new_emotion}")
            return
    print(f"id {clip_id} not found")

with open(MANIFEST, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)
print("manifest saved")

In [ ]:
from datasets import Dataset, Audio, Features, Value, concatenate_datasets, load_dataset
from huggingface_hub import login

login(token=HF_TOKEN)

clean = [
    m for m in manifest
    if 5 <= m["duration_sec"] <= 120
    and len(m["transcript"].split()) >= 5
    and m["emotion"] != ""
]
print(f"new clips to append: {len(clean)}  ({len(manifest)-len(clean)} removed by QC)")

print("loading existing dataset from HF...")
existing_ds = load_dataset(HF_REPO, split="train", token=HF_TOKEN)
print(f"   existing rows    : {len(existing_ds)}")

print("reading audio files into bytes...")
audio_structs = []
for m in clean:
    with open(m["audio_path"], "rb") as f:
        audio_bytes = f.read()
    audio_structs.append({
        "bytes": audio_bytes,
        "path":  m["audio_path"]
    })

new_data = {
    "id":           [m["id"]           for m in clean],
    "audio":        audio_structs,                         # {bytes, path} struct
    "transcript":   [m["transcript"]   for m in clean],
    "duration_sec": [float(m["duration_sec"]) for m in clean],  # float64 to match
    "emotion":      [m["emotion"]      for m in clean],
    "label_source": [m["label_source"] for m in clean],
    "language":     ["en"              for _ in clean],
    "source":       ["audiobook"       for _ in clean],
}

new_ds = Dataset.from_dict(new_data, features=existing_ds.features)
print(f"   new rows         : {len(new_ds)}")
print(f"   features match   : {new_ds.features == existing_ds.features}")

combined_ds = concatenate_datasets([existing_ds, new_ds])
print(f"   combined total   : {len(combined_ds)}")

combined_ds.push_to_hub(HF_REPO, token=HF_TOKEN, private=False)
print(f" appended → https://huggingface.co/datasets/{HF_REPO}")

from collections import Counter
langs = Counter(combined_ds["language"])
print("\n  language breakdown:")
for lang, cnt in langs.items():
    print(f"    {lang}: {cnt} clips")